In [0]:
silver_df = spark.read.table("workspace.silver.gbif_occurrences")

In [0]:
from pyspark.sql import functions as F


species_summary_df = (
    silver_df
    .groupBy(
        "scientificName",
        "query_scientific_name",
    )
    .agg(
        F.count("*").alias("observations")
    )
    .orderBy(
        F.desc("observations")
    )
)

display(species_summary_df)

scientificName,query_scientific_name,observations
"Hynobius nebulosus (Temminck & Schlegel, 1838)",Hynobius nebulosus,20
"Hynobius lichenatus Boulenger, 1883",Hynobius lichenatus,17
"Hynobius unnangso Tago, 1931",Hynobius lichenatus,3


In [0]:
gold_table = "workspace.gold.species_summary"

(
    species_summary_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(gold_table)
)

print(
    f"Gold table updated: {gold_table}, "
    f"records={species_summary_df.count()}"
)

Gold table updated: workspace.gold.species_summary, records=3


In [0]:
from pyspark.sql import functions as F


country_summary_df = (
    silver_df
    .groupBy(
        "countryCode",
        "query_scientific_name",
    )
    .agg(
        F.count("*").alias("observations")
    )
    .orderBy(
        F.desc("observations")
    )
)

display(country_summary_df)

countryCode,query_scientific_name,observations
JP,Hynobius lichenatus,20
JP,Hynobius nebulosus,20


In [0]:
gold_table = "workspace.gold.country_summary"

(
    country_summary_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(gold_table)
)

print(
    f"Gold table updated: {gold_table}, "
    f"records={country_summary_df.count()}"
)

Gold table updated: workspace.gold.country_summary, records=2


In [0]:
year_summary_df = (
    silver_df
    .filter(
        F.col("event_year").isNotNull()
    )
    .groupBy(
        "event_year",
        "query_scientific_name",
    )
    .agg(
        F.count("*").alias("observations")
    )
    .orderBy(
        "event_year",
        "query_scientific_name",
    )
)

display(year_summary_df)

event_year,query_scientific_name,observations
2017,Hynobius lichenatus,3
2018,Hynobius lichenatus,3
2018,Hynobius nebulosus,9
2020,Hynobius lichenatus,1
2021,Hynobius nebulosus,1
2022,Hynobius lichenatus,3
2022,Hynobius nebulosus,1
2023,Hynobius lichenatus,4
2023,Hynobius nebulosus,2
2024,Hynobius lichenatus,2


In [0]:
gold_table = "workspace.gold.year_summary"

(
    year_summary_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(gold_table)
)

print(
    f"Gold table updated: {gold_table}, "
    f"records={year_summary_df.count()}"
)

Gold table updated: workspace.gold.year_summary, records=12


In [0]:
basis_summary_df = (
    silver_df
    .groupBy(
        "basisOfRecord",
        "query_scientific_name",
    )
    .agg(
        F.count("*").alias("observations")
    )
    .orderBy(
        "query_scientific_name",
        F.desc("observations"),
    )
)

display(basis_summary_df)

basisOfRecord,query_scientific_name,observations
PRESERVED_SPECIMEN,Hynobius lichenatus,9
HUMAN_OBSERVATION,Hynobius lichenatus,7
MATERIAL_SAMPLE,Hynobius lichenatus,4
HUMAN_OBSERVATION,Hynobius nebulosus,10
PRESERVED_SPECIMEN,Hynobius nebulosus,10


In [0]:
gold_table = "workspace.gold.observation_type_summary"

(
    basis_summary_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(gold_table)
)

print(
    f"Gold table updated: {gold_table}, "
    f"records={basis_summary_df.count()}"
)

Gold table updated: workspace.gold.observation_type_summary, records=5


In [0]:
from pyspark.sql import functions as F


geographic_observations_df = (
    silver_df
    .filter(
        F.col("decimalLatitude").isNotNull()
    )
    .filter(
        F.col("decimalLongitude").isNotNull()
    )
    .select(
        "gbifID",
        "scientificName",
        "query_scientific_name",
        "countryCode",
        "decimalLatitude",
        "decimalLongitude",
        "eventDate",
        "event_year",
        "event_month",
        "event_day",
    )
)

display(geographic_observations_df)

gbifID,scientificName,query_scientific_name,countryCode,decimalLatitude,decimalLongitude,eventDate,event_year,event_month,event_day
6236231602,"Hynobius lichenatus Boulenger, 1883",Hynobius lichenatus,JP,38.892438,141.203335,2026-03-30T15:20:22,2026,3,30
6252318317,"Hynobius lichenatus Boulenger, 1883",Hynobius lichenatus,JP,39.792143,141.1084,2026-04-26T13:34,2026,4,26
6334740005,"Hynobius unnangso Tago, 1931",Hynobius lichenatus,JP,37.172208,139.492308,2026-05-17T11:14:34,2026,5,17
6334967347,"Hynobius lichenatus Boulenger, 1883",Hynobius lichenatus,JP,37.319213,139.591118,2026-05-09T12:28:11,2026,5,9
4597022977,"Hynobius unnangso Tago, 1931",Hynobius lichenatus,JP,36.846644,139.730744,2024-03-15T08:00:23,2024,3,15
4863665746,"Hynobius unnangso Tago, 1931",Hynobius lichenatus,JP,37.053815,139.829884,2024-05-11T14:45,2024,5,11
4908235134,"Hynobius lichenatus Boulenger, 1883",Hynobius lichenatus,JP,38.340868,140.704923,2023-06-03T05:21,2023,6,3
6274122513,"Hynobius nebulosus (Temminck & Schlegel, 1838)",Hynobius nebulosus,JP,33.584885,130.274533,2026-01-22T20:37:14,2026,1,22
6234740240,"Hynobius nebulosus (Temminck & Schlegel, 1838)",Hynobius nebulosus,JP,33.41236,130.257765,2026-04-16T20:37,2026,4,16
6398873387,"Hynobius nebulosus (Temminck & Schlegel, 1838)",Hynobius nebulosus,JP,33.693447,130.3581,2026-04-16T19:55,2026,4,16


In [0]:
# 簡易的な分析用のため、緯度経度は0.1度程度にまとめる（近い観察地点をある程度まとめておく）
geographic_summary_df = (
    geographic_observations_df
    .withColumn(
        "latitude_grid",
        F.round(F.col("decimalLatitude"), 1)
    )
    .withColumn(
        "longitude_grid",
        F.round(F.col("decimalLongitude"), 1)
    )
    .groupBy(
        "latitude_grid",
        "longitude_grid",
        "query_scientific_name",
    )
    .agg(
        F.count("*").alias("observations")
    )
    .orderBy(
        "query_scientific_name",
        F.desc("observations"),
    )
)

display(geographic_summary_df)

latitude_grid,longitude_grid,query_scientific_name,observations
37.3,139.6,Hynobius lichenatus,1
36.8,139.7,Hynobius lichenatus,1
38.3,140.7,Hynobius lichenatus,1
37.2,139.5,Hynobius lichenatus,1
37.1,139.8,Hynobius lichenatus,1
39.8,141.1,Hynobius lichenatus,1
38.9,141.2,Hynobius lichenatus,1
33.6,130.3,Hynobius nebulosus,2
33.9,130.7,Hynobius nebulosus,2
33.1,129.9,Hynobius nebulosus,1


In [0]:
gold_table = "workspace.gold.geographic_summary"

(
    geographic_summary_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(gold_table)
)

print(
    f"Gold table updated: {gold_table}, "
    f"records={geographic_summary_df.count()}"
)

Gold table updated: workspace.gold.geographic_summary, records=15
